Weekly Sales By Store and Holiday

In [ ]:
from snowflake.snowpark.context import get_active_session
import plotly.express as px
import streamlit as st

# Get the current active session context
session = get_active_session()

snowpark_df = session.sql(
    """
    SELECT STORE_ID, ISHOLIDAY ,SUM(WEEKLY_SALES) AS TOTAL_WEEKLY_SALES
    FROM WALMART_FACT_VIEW 
    GROUP BY ISHOLIDAY,STORE_ID
    """
)

my_df = snowpark_df.to_pandas()
#print(my_df)

#PAss dataframe to the plotly chanrt

#fig = px.bar(my_df,x=['STORE_ID','ISHOLIDAY'],y='WEEKLY_SALES', title = "Weekly Slaes By Store and Holidays" )
fig = px.bar(my_df, 
             x='STORE_ID', 
             y='TOTAL_WEEKLY_SALES', 
             color='ISHOLIDAY', 
             barmode='group',
             title="Weekly Sales By Store and Holidays")

# 2. Use Streamlit to display it
st.plotly_chart(fig)

Weekly Sales By Temperature and Year

In [ ]:
from snowflake.snowpark.context import get_active_session
import plotly.express as px
import plotly.graph_objects as go
import streamlit as st

#Get active session context
session = get_active_session()

# 1. SQL: Aggregate Sales by Year and Temperature
# We use FLOOR to 'bin' temperature so the X-axis isn't too crowded
sql_query = """
WITH BinnedSales AS (
    SELECT 
        -- 1. Create temperature bins (e.g., bin '20' means 20-29 degrees)
        FLOOR(TEMPERATURE / 10) * 10 AS TEMP_BIN,
        YEAR(TO_DATE(DATE)) AS YEAR_VAL,
        SUM(WEEKLY_SALES) AS TOTAL_SALES
    FROM WALMART_FACT_VIEW
    GROUP BY TEMP_BIN, YEAR_VAL
),
SalesDiff AS (
    SELECT 
        TEMP_BIN,
        YEAR_VAL,
        TOTAL_SALES,
        -- 2. Find the previous bin's sales to calculate the change
        LAG(TOTAL_SALES) OVER (ORDER BY YEAR_VAL, TEMP_BIN) AS PREV_SALES
    FROM BinnedSales
)

SELECT 
    -- 3. Final labels for your X-axis (e.g., "2010 - 20s")
    CONCAT(YEAR_VAL, ' - ', TEMP_BIN, 's') AS LABEL,
    -- The value for the waterfall (first value is absolute, others are relative)
    CASE 
        WHEN PREV_SALES IS NULL THEN TOTAL_SALES 
        ELSE TOTAL_SALES - PREV_SALES 
    END AS SALES_CHANGE,
    -- Measure type for Plotly: 'absolute' for the first bar, 'relative' for changes
    CASE 
        WHEN PREV_SALES IS NULL THEN 'absolute' 
        ELSE 'relative' 
    END AS MEASURE_TYPE
FROM SalesDiff
ORDER BY YEAR_VAL, TEMP_BIN;
"""

df_waterfall = session.sql(sql_query).to_pandas()

fig = go.Figure(go.Waterfall(
    name = "Sales Variance",
    orientation = "v",
    measure = df_waterfall['MEASURE_TYPE'],
    x = df_waterfall['LABEL'],
    y = df_waterfall['SALES_CHANGE'],
    connector = {"line":{"color":"rgb(63, 63, 63)"}},
))

fig.update_layout(
    title = "Weekly Sales by Temperature and Year",
    showlegend = True
)
st.plotly_chart(fig)

Weekly Sales By Store Size

In [ ]:
import plotly.express as px
import streamlit as st
from snowflake.snowpark.context import get_active_session

session = get_active_session()

query = """
SELECT  store_size ,
SUM(WEEKLY_SALES) AS WEEKLY_SALES
FROM WALMART_FACT_VIEW 
GROUP BY store_size
ORDER BY STORE_SIZE
"""

df = session.sql(query).to_pandas()

# 1. Ensure your data is sorted by 'Size' so the area fills correctly
#df_sorted = df.sort_values('SIZE')

# 2. Create the Area Chart
fig = px.area(
    df, 
    x='STORE_SIZE', 
    y='WEEKLY_SALES',
    title='Weekly Sales by Store Size',
    labels={'SIZE': 'Store Size', 'WEEKLY_SALES': 'Weekly Sales'}
)

# 3. Match the visual style (optional)
fig.update_traces(line_color='#0072B2', fillcolor='rgba(0, 114, 178, 0.3)')

st.plotly_chart(fig)

Weekly Sales By Store Type And Month

In [ ]:
import plotly.express as px
import streamlit as st
from snowflake.snowpark.context import get_active_session

session = get_active_session()

query = """
SELECT  
STORE_TYPE ,
MONTH(TO_DATE(DATE)) AS MONTH,
MONTHNAME(TO_DATE(DATE)) AS MONTHNAME,
SUM(WEEKLY_SALES) AS TOTAL_WEEKLY_SALES
FROM WALMART_FACT_VIEW 
GROUP BY MONTH,STORE_TYPE,MONTHNAME
ORDER BY MONTH
"""

df = session.sql(query).to_pandas()

fig = px.line(
    df, 
    x='MONTHNAME', 
    y='TOTAL_WEEKLY_SALES', 
    color='STORE_TYPE',     # Creates the separate colored lines
    markers=True,            # Adds the dots at each month
    text='TOTAL_WEEKLY_SALES',   # Adds the numeric labels
    title='Weekly Sales by Store Type and Month'
)

# 3. Clean up the labels (rounding and adding 'M')
#fig.update_traces(texttemplate='%{text:.0f}M', textposition='top center')

# 4. Final layout tweaks to match the image
fig.update_layout(
    yaxis_title="Weekly Sales",
    xaxis_title="Month",
    legend_title="Store Type"
)

st.plotly_chart(fig)



Markdown Sales By Year And Store

In [ ]:
import plotly.express as px
import streamlit as st
from snowflake.snowpark.context import get_active_session

session = get_active_session()

query = """
SELECT
STORE_ID,
YEAR(TO_DATE(DATE)) AS YEAR,
ZEROIFNULL(SUM(TRY_CAST(NULLIF(MARKDOWN1, 'NA') AS FLOAT))) as MARKDOWN1,
ZEROIFNULL(SUM(TRY_CAST(NULLIF(MARKDOWN2, 'NA') AS FLOAT))) as MARKDOWN2,
ZEROIFNULL(SUM(TRY_CAST(NULLIF(MARKDOWN3, 'NA') AS FLOAT))) as MARKDOWN3,
ZEROIFNULL(SUM(TRY_CAST(NULLIF(MARKDOWN4, 'NA') AS FLOAT))) as MARKDOWN4,
ZEROIFNULL(SUM(TRY_CAST(NULLIF(MARKDOWN5, 'NA') AS FLOAT))) as MARKDOWN5
FROM WALMART_FACT_VIEW
GROUP BY STORE_ID, YEAR
"""

df = session.sql(query).to_pandas()

fig = px.bar(
    df, 
    x='YEAR', 
    y=['MARKDOWN1', 'MARKDOWN2', 'MARKDOWN3', 'MARKDOWN4', 'MARKDOWN5'],
    barmode='group',
    title="Markdown Sales By Year and Store",
    labels={'value': 'Sales Amount', 'variable': 'Markdown Type'}
)

# 4. Format Y-Axis to show Millions (or Billions like your image)
# To show Millions ($M), we divide or use tickformat
fig.update_layout(yaxis_tickformat='.2s') # '.2s' uses SI prefixes like 'M' or 'G'
fig.update_traces(textposition='outside')

st.plotly_chart(fig)


Weekly Sales By StoreType

In [ ]:
from snowflake.snowpark.context import get_active_session
import plotly.express as px
import streamlit as st

session = get_active_session()

Pie_Query = """
SELECT  
STORE_TYPE ,
SUM(WEEKLY_SALES) AS TOTAL_WEEKLY_SALES
FROM WALMART_FACT_VIEW 
GROUP BY STORE_TYPE
"""

Bar_Query = """
SELECT  
STORE_TYPE ,STORE_ID,
SUM(WEEKLY_SALES) AS TOTAL_WEEKLY_SALES
FROM WALMART_FACT_VIEW 
GROUP BY STORE_TYPE,STORE_ID
"""

pie_df = session.sql(Pie_Query).to_pandas()
bar_df = session.sql(Bar_Query).to_pandas()
# 1. Force the columns to be strings so Plotly treats them as labels, not numbers
bar_df['STORE_ID'] = bar_df['STORE_ID'].astype(str)
bar_df['STORE_TYPE'] = bar_df['STORE_TYPE'].astype(str)

fig_pie = px.pie(pie_df,
            values = 'TOTAL_WEEKLY_SALES',
            names = 'STORE_TYPE',
            title ='Weekly Sales By Store Type')


fig_bar = px.bar(bar_df,
                x='TOTAL_WEEKLY_SALES',
                y='STORE_ID',
                facet_row='STORE_TYPE', # This creates the sections for A, B, and C
                orientation = 'h',
                color='STORE_ID',
                text_auto='.3s',
                title='Weekly Sales By Store Type And Store ID')

# 3. Create the buttons
# We use 'update' method to toggle the 'visible' state of each bar trace
types = sorted(bar_df['STORE_TYPE'].unique())
buttons = [
    dict(label="All Types",
         method="update",
         args=[{"visible": [True] * len(bar_df)}, # Show all bars
               {"title": "All Store Types"}])
]

for store_type in types:
    # A list of booleans: True if the row matches the selected type
    visibility = [t == store_type for t in bar_df['STORE_TYPE']]
    buttons.append(
        dict(label=f"Type {store_type}",
             method="update",
             args=[{"visible": visibility}, 
                   {"title": f"Sales for Type {store_type}"}])
    )

fig_bar.update_layout(
    # 1. Adjust the Main Title position
    title={
        'text': "Weekly Sales By Store Type And Store ID",
        'y': 0.98,          # Push title to the very top
        'x': 0.5,           # Center the title
        'xanchor': 'center',
        'yanchor': 'top'
    },

# 2. Adjust Dropdown position
    updatemenus=[
        dict(
            type="dropdown",
            direction="down",
            x=0.0,          # Move to the far left
            y=1.1,          # Position it below the title but above the chart
            showactive=True,
            buttons=buttons
        )
    ],

# 3. Add Top Margin to prevent squeezing
    margin=dict(t=150, l=100), 
    height=800,
    showlegend=False,
    template='plotly_white'
)

# 5. Apply the axis cleaning from the previous step
fig_bar.update_yaxes(showticklabels=False, title_text="")
fig_bar.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1], x=-0.02))

st.plotly_chart(fig_pie)
st.plotly_chart(fig_bar)

Fuel Price By YEar

In [ ]:
from snowflake.snowpark.context import get_active_session
import plotly.express as px
import streamlit as st
import plotly.graph_objects as go

session = get_active_session()

Pie_Query = """
SELECT 
SUM(FUEL_PRICE) AS YEARLY_FUEL_PRICE,
YEAR(TO_DATE(DATE)) AS YEAR,
STORE_ID 
FROM WALMART_FACT_VIEW 
GROUP BY YEAR,STORE_ID
"""

pie_df = session.sql(Pie_Query).to_pandas()

fig_donut = px.pie(pie_df, 
                   values='YEARLY_FUEL_PRICE', 
                   names='YEAR', 
                   hole=0.5, # This creates the 'donut' hole
                   title='Fuel Price by Year')

st.plotly_chart(fig_donut)

# 1. Pivot and Total in one step using Pandas
# This automatically handles the grouping and sums the values
df_pivot = pie_df.pivot_table(index='STORE_ID', 
                              columns='YEAR', 
                              values='YEARLY_FUEL_PRICE', 
                              aggfunc='sum').reset_index()

# 2. Add the Total column (axis=1 sums horizontally across years)
df_pivot['Total'] = df_pivot.iloc[:, 1:].sum(axis=1)

# 3. Create the Plotly Table
fig_table = go.Figure(data=[go.Table(
    header=dict(values=list(df_pivot.columns), 
                fill_color='lightgrey', 
                align='left'),
    cells=dict(values=[df_pivot[col] for col in df_pivot.columns],
               format=[None] + [",.2f"] * (len(df_pivot.columns)-1), # Formats numbers with commas
               fill_color='white',
               align='left'))
])


st.plotly_chart(fig_table)

WEEKLY SALES BY YEAR, MONTH AND DATE

In [ ]:
from snowflake.snowpark.context import get_active_session
import plotly.express as px
import streamlit as st

session = get_active_session()

###YEAR 
year_query = """
SELECT  
YEAR(TO_DATE(DATE)) AS YEAR,
SUM(WEEKLY_SALES) AS TOTAL_WEEKLY_SALES
FROM WALMART_FACT_VIEW 
GROUP BY YEAR"""

###MONTH 
month_query = """SELECT  
MONTHNAME(TO_DATE(DATE)) AS MONTH,
SUM(WEEKLY_SALES) AS TOTAL_WEEKLY_SALES
FROM WALMART_FACT_VIEW 
GROUP BY MONTH"""

###DATE
day_query = """SELECT  
DAY(TO_DATE(DATE)) AS DAY,
SUM(WEEKLY_SALES) AS TOTAL_WEEKLY_SALES
FROM WALMART_FACT_VIEW 
GROUP BY DAY
ORDER BY DAY"""

year_df = session.sql(year_query).to_pandas()
month_df = session.sql(month_query).to_pandas()
day_df = session.sql(day_query).to_pandas()

## Year chart
year_fig = px.bar(year_df,
                 x='YEAR',
                 y='TOTAL_WEEKLY_SALES',
                 barmode='group',
                 title="Weekly Sales By Year"
                 )
year_fig.update_layout(bargap=0.7)

## Month chart
month_fig = px.bar(month_df,
                 x='MONTH',
                 y='TOTAL_WEEKLY_SALES',
                 barmode='group',
                 title="Weekly Sales By Month")

# ## Date chart
day_fig = px.bar(day_df,
                 x='DAY',
                 y='TOTAL_WEEKLY_SALES',
                 barmode='group',
                 title="Weekly Sales By Day")

st.plotly_chart(year_fig)
st.plotly_chart(month_fig)
st.plotly_chart(day_fig)



Weekly Sales By CPI

In [ ]:
from snowflake.snowpark.context import get_active_session
import plotly.express as px
import pandas as pd
import streamlit as st

session = get_active_session()

###YEAR 
query = """
SELECT 
CPI,
SUM(WEEKLY_SALES) AS TOTAL_WEEKLY_SALES
FROM WALMART_FACT_VIEW 
GROUP BY CPI"""

cpi_df = session.sql(query).to_pandas()

cpi_fig = px.line(cpi_df,
                 x='CPI',
                 y='TOTAL_WEEKLY_SALES',
                 title='Weekly Sales By CPI')

# 1. Force CPI to be a number to match the x axsis as per the project (this is the most important step)
cpi_df['CPI'] = pd.to_numeric(cpi_df['CPI'], errors='coerce')

# 2. Sort by CPI so the line connects points from left to right otherwise it looks clumsy
cpi_df = cpi_df.sort_values('CPI')

# 3. Create the chart
cpi_fig = px.line(cpi_df, 
              x='CPI', 
              y='TOTAL_WEEKLY_SALES', 
              title='Weekly Sales By CPI')

# 4. Match the dotted style from your original image
cpi_fig.update_traces(line=dict(dash='dot', width=1))

# 5. Clean up the X-axis so it shows a clean numeric range (120, 140, 160...)
cpi_fig.update_xaxes(type='linear')

st.plotly_chart(cpi_fig)

Department Wise Weekly Sales

In [ ]:
from snowflake.snowpark.context import get_active_session
import plotly.express as px
import pandas as pd
import streamlit as st

session = get_active_session()

###YEAR 
query = """
SELECT 
DEPT_ID,
SUM(WEEKLY_SALES) AS TOTAL_WEEKLY_SALES
FROM WALMART_FACT_VIEW 
GROUP BY DEPT_ID"""

df = session.sql(query).to_pandas()

fig = px.bar(df,
              x='DEPT_ID',
              y='TOTAL_WEEKLY_SALES',
              color ='DEPT_ID',
              barmode='relative',
              title="Weekly Sales By Department"
                 )
st.plotly_chart(fig)
########################## createing pivot table
pivot_df = df.pivot_table(index='DEPT_ID',
                         values='TOTAL_WEEKLY_SALES',
                         aggfunc='sum').reset_index()
#print(pivot_df)
# Using the exact uppercase names from your image
fig_table = go.Figure(data=[go.Table(
    header=dict(
        values=["<b>Dept ID</b>", "<b>Total Weekly Sales</b>"],
        fill_color='#4d4d4d',
        font=dict(color='white', size=12),
        align='left'
    ),
    cells=dict(
        values=[
            pivot_df['DEPT_ID'], 
            pivot_df['TOTAL_WEEKLY_SALES']
        ],
        # Alternating row colors: light grey and white
        fill_color=[['#f2f2f2', 'white'] * 41], 
        align='left',
        # This format converts '5.56e+09' to '5,563,745,000.00'
        format=[None, ",.2f"] 
    )
)])

fig_table.update_layout(height=400, margin=dict(l=0, r=0, t=10, b=10))

st.plotly_chart(fig_table)

###############Find top 5 dept weekly sales 
#1  Sort to find top 5 weekly sales by department
top_5_df = df.sort_values(by='TOTAL_WEEKLY_SALES', ascending=False).head(5)

# 2. Create the minimalist table
fig_top5_table = go.Figure(data=[go.Table(
    # Column widths: narrow for ID, wider for Sales
    columnwidth=[20, 80],
    
    # Empty header to remove the top bar
    header=dict(values=["", ""], height=0, line_color='rgba(0,0,0,0)', fill_color='rgba(0,0,0,0)'),
    
    cells=dict(
        values=[top_5_df['DEPT_ID'], top_5_df['TOTAL_WEEKLY_SALES']],
        
        # Alignment: ID to the left, Sales to the right
        align=['left', 'right'],
        
        # Formatting
        line_color='rgba(0,0,0,0)', # Makes cell borders transparent
        fill_color='white',
        font=dict(family="Arial", size=14, color="black"),
        height=40,
        
        # Formats numbers with commas and 2 decimals
        format=[None, ",.2f"] 
    )
)])

# 3. Add the "Top 5" title manually as an annotation
fig_top5_table.update_layout(
    title="Top 5 department wise sales",
    height=300,
    margin=dict(l=10, r=10, t=50, b=10)
)

#print(top_5_df)

st.plotly_chart(fig_top5_table)